In [2]:
!nvidia-smi

Wed Nov 18 07:45:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from google.colab import files

uploaded = files.upload()

Saving dataset.zip to dataset.zip


In [4]:
!unzip -q /content/dataset.zip -d /content/

In [5]:
!ls -lh /content/dataset

total 16K
-rw-rw-r-- 1 root root   92 Nov 18 07:36 data.yaml
drwxrwxr-x 4 root root 4.0K Nov 18 07:36 test
drwxrwxr-x 4 root root 4.0K Nov 18 07:36 train
drwxrwxr-x 4 root root 4.0K Nov 18 07:36 valid


In [6]:
!cat /content/dataset/data.yaml

train: train/images
val: valid/images
test: test/images

nc: 3
names: ['cat', 'dog', 'car']


In [7]:
!pip install -q ultralytics

In [8]:
import torch
import ultralytics

print("Ultralytics 버전:", ultralytics.__version__)
print("PyTorch 버전:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 버전: 8.4.215
PyTorch 버전: 2.14.1+cu130
CUDA 사용 가능: True
GPU: Tesla T4


In [9]:
from ultralytics import YOLO

# 사전 학습된 YOLOv8n 모델 불러오기
model = YOLO("yolov8n.pt")

In [10]:
results = model.train(
    data="/content/dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    name="homework2_yolov8n"
)

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=homework2_yolov8n, nbs=64, nms=None, opset=None, optimize=Fal

In [11]:
from ultralytics import YOLO

# 가장 성능이 좋았던 모델 불러오기
best_model = YOLO(
    "/content/runs/detect/homework2_yolov8n/weights/best.pt"
)

# 테스트 데이터 평가
test_results = best_model.val(
    data="/content/dataset/data.yaml",
    split="test",
    device=0
)

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,133,193 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1174.9±632.1 MB/s, size: 55.4 KB)
val: Scanning /content/dataset/test/labels... 64 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 64/64 1.8Kit/s 0.0s
val: New cache created: /content/dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.5it/s 2.8s
                   all         64         74      0.861      0.751      0.814      0.512
                   cat         18         18      0.684      0.611      0.597      0.255
                   dog         22         22      0.963      0.773      0.924      0.536
                   car         24         34      0.937      0.869      0.921      0.745
Speed: 4.6ms preprocess, 8.6ms inference, 0.0ms loss, 7.5ms postprocess per image
Results s

In [14]:
from google.colab import files

uploaded = files.upload()

Saving combined_video.mp4 to combined_video.mp4


In [15]:
import os

print(os.path.exists("/content/combined_video.mp4"))

True


In [16]:
from ultralytics import YOLO

best_model = YOLO(
    "/content/runs/detect/homework2_yolov8n/weights/best.pt"
)

print("모델 로드 완료")

모델 로드 완료


In [17]:
track_results = best_model.track(
    source="/content/combined_video.mp4",
    tracker="bytetrack.yaml",
    conf=0.25,
    save=True,
    device=0
)

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 2 packages in 179ms
Prepared 1 package in 40ms
Installed 1 package in 1ms
 + lap==0.5.14

requirements: AutoUpdate success ✅ 0.5s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/150) /content/combined_video.mp4: 384x640 (no detections), 6.7ms
video 1/1 (frame 2

In [18]:
from IPython.display import Video, display

display(Video(
    "/content/runs/detect/track/combined_video.avi",
    embed=True
))

In [19]:
import os

video_path = "/content/runs/detect/track/combined_video.avi"

print("파일 존재 여부:", os.path.exists(video_path))

if os.path.exists(video_path):
    print("파일 크기:", os.path.getsize(video_path), "bytes")

파일 존재 여부: True
파일 크기: 25707352 bytes


In [20]:
!ffmpeg -i /content/runs/detect/track/combined_video.avi \
-c:v libx264 -pix_fmt yuv420p \
-movflags +faststart \
-y /content/combined_video_tracked.mp4

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [21]:
from IPython.display import Video, display

display(Video(
    "/content/combined_video_tracked.mp4",
    embed=True
))

In [22]:
from google.colab import files

# 학습된 모델 다운로드
files.download(
    "/content/runs/detect/homework2_yolov8n/weights/best.pt"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
files.download("/content/combined_video_tracked.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>